# 28. JAX — `jax`
**Python Libraries Lab · Phase 2 · Deep Learning & AI**

_NumPy with automatic gradients, compilation and vectorisation._

✅ **Runs in Colab** as-is.

JAX looks like NumPy, but adds three powerful function transformations. grad turns a function into one that returns its gradient. jit compiles a function so it runs much faster. vmap takes a function written for one example and runs it over a whole batch. Google DeepMind uses JAX for large research models.

**In this notebook:** You will use grad to differentiate functions, speed up a calculation with jit, use vmap to batch a function, and then fit a straight line to ShopEasy ad spend and sales with gradient descent written from scratch.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
JAX is pre-installed in Colab.

## Step 1 · NumPy, but JAX
`jax.numpy` mirrors NumPy. Arrays cannot be changed in place.

In [ ]:
import jax, jax.numpy as jnp
print("JAX", jax.__version__, "| devices:", jax.devices())

x = jnp.arange(5.0)
print(x * 2, jnp.sin(x).round(2), x.mean())
y = x.at[0].set(100.0)      # returns a NEW array
print("original:", x, "| new:", y)

## Step 2 · grad: derivatives automatically
`grad(f)` is a new function that returns df/dx.

In [ ]:
from jax import grad, jit, vmap

def f(x):
    return x ** 3 + 2 * x
df = grad(f)
print("f'(2) =", df(2.0))          # 3*4 + 2 = 14

def profit(price):
    return (price - 300) * 1000 * jnp.exp(-price / 800)
print("Slope of profit at price 900:", grad(profit)(900.0))

## Step 3 · jit: compile for speed
The first call compiles; later calls are fast.

In [ ]:
import time

def slow_sum(x):
    return jnp.sum(jnp.tanh(x) ** 2 + jnp.cos(x) * jnp.sin(x))
fast_sum = jit(slow_sum)

big = jnp.linspace(0, 10, 5_000_000)
fast_sum(big).block_until_ready()      # compile once
t = time.perf_counter(); slow_sum(big).block_until_ready(); t1 = time.perf_counter() - t
t = time.perf_counter(); fast_sum(big).block_until_ready(); t2 = time.perf_counter() - t
print(f"without jit {t1*1000:.1f} ms | with jit {t2*1000:.1f} ms")

## Step 4 · vmap: one example to a whole batch
Write the function for one order; `vmap` runs it on all of them.

In [ ]:
def order_total(qty, price):
    return qty * price * jnp.where(qty >= 3, 0.9, 1.0)   # 10% off for 3+ items

qtys = jnp.array([1, 3, 5, 2])
prices = jnp.array([499.0, 199.0, 999.0, 1499.0])
print(vmap(order_total)(qtys, prices))

## Step 5 · Mini project: fit a line with gradient descent
Sales = w × ad spend + b. `grad` gives the slope of the error; we step downhill 300 times.

In [ ]:
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
ads = jax.random.uniform(k1, (100,), minval=1, maxval=10)        # Rs. thousands
sales = 3.0 * ads + 5.0 + jax.random.normal(k2, (100,))           # true w=3, b=5

def loss(params):
    w, b = params
    return jnp.mean((w * ads + b - sales) ** 2)

@jit
def step(params, lr=0.02):
    g = grad(loss)(params)
    return params - lr * g

params = jnp.array([0.0, 0.0])
for i in range(1500):
    params = step(params)
print("learned w, b:", params.round(2), "| loss:", round(float(loss(params)), 3))

## ✍️ Your turn
Use `grad` to find the derivative of `g(x) = jnp.sin(x) * x` at `x = 0.0`. Store it as a Python float in `answer` (it should be 0.0).

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer is not None and abs(answer) < 1e-6, f"Expected 0.0, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = float(grad(lambda x: jnp.sin(x) * x)(0.0))

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **jax.numpy**
- **grad**
- **jit**
- **vmap**
- **Pure function**
- **PRNG key**

**Next:** 29. Hugging Face Transformers (`transformers`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/29_transformers.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install jax
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.